# UCSF Precision Oncology — HNSCC Analysis

This notebook demonstrates a data scientist workflow for analyzing HNSCC (Head and Neck Squamous Cell Carcinoma) patient data across three governed data collections:

1. **HNSCC Patient Cohort** — 200 patients with clinical data
2. **Tumor Genomics** — somatic mutation profiles
3. **Drug-Target Reference** — approved therapies and clinical trials

**Goal:** Identify patients with actionable mutations who may benefit from targeted therapies.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from google.cloud import bigquery

client = bigquery.Client()

PATIENT_TABLE = "wb-potent-shallot-9879.hnscc_clinical_data.patients"
MUTATION_TABLE = "wb-twinkly-banana-2547.hnscc_genomics_data.somatic_mutations"
DRUG_TABLE = "wb-sunny-pecan-1742.hnscc_drug_target_data.drug_targets"
TRIAL_TABLE = "wb-sunny-pecan-1742.hnscc_drug_target_data.clinical_trials"

print("Setup complete.")

## 1. Load Patient Cohort Data

In [ ]:
patients = client.query(f"SELECT * FROM `{PATIENT_TABLE}`").to_dataframe()
print(f"Loaded {len(patients)} patients")
print(f"\nCohort Summary:")
print(f"  Median age: {patients['age'].median():.0f} years")
print(f"  Sex: {patients['sex'].value_counts().to_dict()}")
print(f"  Stage distribution: {patients['stage'].value_counts().sort_index().to_dict()}")
print(f"  HPV positive: {(patients['hpv_status'] == 'Positive').sum()} ({(patients['hpv_status'] == 'Positive').mean()*100:.0f}%)")
patients.head()

## 2. Load Somatic Mutation Data

In [ ]:
mutations = client.query(f"SELECT * FROM `{MUTATION_TABLE}`").to_dataframe()
print(f"Loaded {len(mutations)} mutation records across {mutations['patient_id'].nunique()} patients")
print(f"\nGenes mutated: {mutations['gene'].nunique()}")
print(f"Most common: {mutations['gene'].value_counts().head(5).to_dict()}")
mutations.head()

## 3. Mutation Frequency Across the Cohort

In [ ]:
n_patients = len(patients)
gene_freq = (
    mutations.groupby('gene')['patient_id']
    .nunique()
    .sort_values(ascending=True)
    .reset_index()
)
gene_freq.columns = ['Gene', 'Patients']
gene_freq['Percentage'] = (gene_freq['Patients'] / n_patients * 100).round(1)

fig = px.bar(
    gene_freq, x='Percentage', y='Gene', orientation='h',
    text=gene_freq.apply(lambda r: f"{r['Percentage']}% ({r['Patients']} pts)", axis=1),
    title='Gene Mutation Frequency in HNSCC Cohort',
    labels={'Percentage': '% of Cohort'},
    color='Percentage',
    color_continuous_scale='Blues',
)
fig.update_traces(textposition='outside')
fig.update_layout(height=600, showlegend=False, coloraxis_showscale=False)
fig.show()

## 4. HPV-Positive vs HPV-Negative Survival Comparison

In [ ]:
fig = go.Figure()

for status, color in [('Positive', '#2ca02c'), ('Negative', '#d62728')]:
    subset = patients[patients['hpv_status'] == status]
    fig.add_trace(go.Box(
        y=subset['os_months'], name=f'HPV {status} (n={len(subset)})',
        marker_color=color, boxmean=True
    ))

fig.update_layout(
    title='Overall Survival by HPV Status',
    yaxis_title='OS (months)', height=450
)
fig.show()

hpv_summary = patients.groupby('hpv_status').agg(
    n=('patient_id', 'count'),
    median_os=('os_months', 'median'),
    median_pfs=('pfs_months', 'median'),
).round(1)
print("HPV Status Summary:")
hpv_summary

## 5. RET Mutation Discovery

RET mutations are rare but highly actionable — **Selpercatinib** (FDA-approved 2020) specifically targets RET fusions and mutations.

In [ ]:
ret_mutations = mutations[mutations['gene'] == 'RET']
ret_patient_ids = ret_mutations['patient_id'].unique()
ret_patients = patients[patients['patient_id'].isin(ret_patient_ids)]

print(f"Found {len(ret_patient_ids)} patients with RET mutations\n")

print("RET Variants Found:")
print(ret_mutations[['patient_id', 'variant', 'allele_frequency', 'clinical_significance']].to_string(index=False))

print(f"\nRET Patient Demographics:")
print(f"  Age range: {ret_patients['age'].min()}-{ret_patients['age'].max()} (median {ret_patients['age'].median():.0f})")
print(f"  Stage: {ret_patients['stage'].value_counts().to_dict()}")
print(f"  HPV status: {ret_patients['hpv_status'].value_counts().to_dict()}")

In [ ]:
print("RET-Mutated Patients — Treatment History & Outcomes\n")
ret_details = ret_patients[['patient_id', 'age', 'sex', 'stage', 'treatment', 'response', 'pfs_months', 'os_months']].copy()
ret_details

## 6. Drug-Target Matching

Cross-reference RET mutations with the drug-target database to identify targeted therapy candidates.

In [ ]:
drug_targets = client.query(f"SELECT * FROM `{DRUG_TABLE}`").to_dataframe()

ret_drugs = drug_targets[drug_targets['target_gene'] == 'RET']
print(f"Drugs targeting RET ({len(ret_drugs)} found):\n")
ret_drugs[['drug_name', 'mechanism', 'indication', 'fda_status']]

In [ ]:
# Check which RET patients are NOT already receiving targeted therapy
ret_on_targeted = ret_patients[ret_patients['treatment'].str.contains('Selpercatinib|Pralsetinib|Cabozantinib', case=False, na=False)]
ret_no_targeted = ret_patients[~ret_patients['patient_id'].isin(ret_on_targeted['patient_id'])]

print(f"RET-mutated patients currently receiving RET-targeted therapy: {len(ret_on_targeted)}")
print(f"RET-mutated patients NOT on targeted therapy: {len(ret_no_targeted)}")
print(f"\nThese {len(ret_no_targeted)} patients may be candidates for:")
print("  - Selpercatinib (FDA-approved for RET-altered thyroid/NSCLC)")
print("  - Tumor-agnostic basket trial enrollment based on RET mutation status")

## 7. Clinical Trial Matching for RET Patients

In [ ]:
trials = client.query(f"SELECT * FROM `{TRIAL_TABLE}`").to_dataframe()

ret_trials = trials[
    trials['drug'].str.contains('Selpercatinib|Pralsetinib|BLU-667', case=False, na=False) |
    trials['eligibility_criteria'].str.contains('RET', case=False, na=False)
]
recruiting = ret_trials[ret_trials['enrollment_status'] == 'Recruiting']

print(f"Clinical trials relevant to RET-mutated patients: {len(ret_trials)}")
print(f"Currently recruiting: {len(recruiting)}\n")
ret_trials[['trial_id', 'drug', 'phase', 'cancer_type', 'enrollment_status']]

## 8. Summary: Actionable Findings

### Key Findings

1. **RET mutations identified in ~5% of the cohort** (10 patients) — a rare but highly actionable molecular target
2. **Selpercatinib** is FDA-approved for RET-altered thyroid and NSCLC — these HNSCC patients may be candidates for clinical trials or tumor-agnostic basket trials
3. **All 10 RET-mutated patients** are not currently receiving RET-targeted therapy
4. **Multiple recruiting trials** are available for RET-altered solid tumors

### Other Actionable Mutations in the Cohort
- **HRAS** (~5%) → Tipifarnib (breakthrough therapy designation for HRAS-mutated HNSCC)
- **PIK3CA** (~20%) → Alpelisib (FDA-approved for PIK3CA-mutated breast cancer; HNSCC trials active)
- **CDKN2A** (~25%) → CDK4/6 inhibitors (Palbociclib + Cetuximab trials in HNSCC)

In [ ]:
actionable_genes = {'RET': 'Selpercatinib', 'HRAS': 'Tipifarnib', 'PIK3CA': 'Alpelisib', 'CDKN2A': 'Palbociclib'}

summary_rows = []
for gene, drug in actionable_genes.items():
    gene_pids = mutations[mutations['gene'] == gene]['patient_id'].unique()
    gene_pts = patients[patients['patient_id'].isin(gene_pids)]
    n_trials = len(trials[
        trials['drug'].str.contains(drug, case=False, na=False) |
        trials['eligibility_criteria'].str.contains(gene, case=False, na=False)
    ])
    summary_rows.append({
        'Gene': gene,
        'Patients': len(gene_pids),
        '% of Cohort': f"{len(gene_pids)/n_patients*100:.0f}%",
        'Lead Drug': drug,
        'Relevant Trials': n_trials,
    })

summary_df = pd.DataFrame(summary_rows)
print("Actionable Mutation Summary")
print("=" * 70)
summary_df